# 15 · Writing Fast Python

*Contest essentials* · 8 exercises

Estimate the cost first, then pick the Python idiom that fits the time limit.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-15) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-15)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import bisect
import math
from collections import deque

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 15.1 · Will it pass? Estimate before you code  `Core`

**Task.** Write `fits(n, cls)` returning True if an algorithm of class `cls` ("1", "logn", "n", "nlogn", "n2", "2n") on input size n needs at most 10^7 basic steps, a safe budget for one second of Python.

**Example.** `fits(10**5, "nlogn")` -> `True`; `fits(10**5, "n2")` -> `False`; `fits(20, "2n")` -> `True`

**Big O.** O(1).

**Note.** Compiled languages manage roughly 10^8 to 10^9 simple steps per second; CPython manages roughly 10^7. Pick the algorithm with that in mind.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-15-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-15-1)

In [ ]:
# 15.1 Will it pass? Estimate before you code
def fits(n, cls):
    raise NotImplementedError

In [ ]:
def test_ex_15_1():
    assert fits(10 ** 5, 'nlogn')
    assert not fits(10 ** 5, 'n2')
    assert fits(20, '2n')
    assert not fits(30, '2n')
    assert fits(10 ** 18, 'logn')


test_ex_15_1()
print('15.1 passed')

### 15.2 · Sets for membership  `Core`

**Task.** Write `count_hits(queries, items)` counting how many queries appear in items. Convert items to a set once.

**Example.** `count_hits([1, 5, 9], [9, 8, 1])` -> `2`

**Big O.** O(N + Q) with a set, O(N * Q) with `in` on a list.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-15-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-15-2)

In [ ]:
# 15.2 Sets for membership
def count_hits(queries, items):
    raise NotImplementedError

In [ ]:
def test_ex_15_2():
    assert count_hits([1, 5, 9], [9, 8, 1]) == 2


test_ex_15_2()
print('15.2 passed')

### 15.3 · Precompute, then answer  `Core`

**Task.** Write `answer_range_sums(arr, queries)` answering each `(l, r)` inclusive sum query from one prefix-sum array.

**Example.** `answer_range_sums([3, 1, 4, 1, 5], [(0, 2), (1, 4)])` -> `[8, 11]`

**Big O.** O(N + Q) instead of O(N * Q).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-15-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-15-3)

In [ ]:
# 15.3 Precompute, then answer
def answer_range_sums(arr, queries):
    raise NotImplementedError

In [ ]:
def test_ex_15_3():
    assert answer_range_sums([3, 1, 4, 1, 5], [(0, 2), (1, 4)]) == [8, 11]


test_ex_15_3()
print('15.3 passed')

### 15.4 · deque instead of pop(0)  `Core`

**Task.** Write `josephus(n, k)`: n people in a circle, every k-th person leaves; return the leaving order. Use `deque.rotate`.

**Example.** `josephus(7, 3)` -> `[3, 6, 2, 7, 5, 1, 4]`

**Big O.** O(N * k) with rotate; a list with `pop(0)` adds O(N) per removal.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-15-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-15-4)

In [ ]:
# 15.4 deque instead of pop(0)
def josephus(n, k):
    raise NotImplementedError

In [ ]:
def test_ex_15_4():
    assert josephus(7, 3) == [3, 6, 2, 7, 5, 1, 4]
    assert josephus(1, 5) == [1]


test_ex_15_4()
print('15.4 passed')

### 15.5 · Sort once, search many times  `Core`

**Task.** Write `count_at_most(values, queries)` returning, for each query q, how many values are <= q. Sort once and use `bisect_right`.

**Example.** `count_at_most([5, 1, 9, 3], [0, 4, 10])` -> `[0, 2, 4]`

**Big O.** O((N + Q) log N) instead of O(N * Q).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-15-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-15-5)

In [ ]:
# 15.5 Sort once, search many times
def count_at_most(values, queries):
    raise NotImplementedError

In [ ]:
def test_ex_15_5():
    assert count_at_most([5, 1, 9, 3], [0, 4, 10]) == [0, 2, 4]


test_ex_15_5()
print('15.5 passed')

### 15.6 · Read big input fast  `Core`

**Task.** Write `big_sum(data)` for input "N, then N integers" that splits the whole text once and sums with `map(int, ...)`, avoiding a Python-level loop per line.

**Example.** `big_sum("3\n10 20\n30")` -> `60`

**Big O.** O(N). On a judge, read the text with `sys.stdin.buffer.read()` (see 1.8).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-15-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-15-6)

In [ ]:
# 15.6 Read big input fast
def big_sum(data):
    raise NotImplementedError

In [ ]:
def test_ex_15_6():
    assert big_sum('3\n10 20\n30') == 60


test_ex_15_6()
print('15.6 passed')

### 15.7 · Build output once  `Core`

**Task.** Write `squares_output(n)` returning the squares of 1..n, one per line, as a single string built with `"\n".join`.

**Example.** `squares_output(3)` -> `"1\n4\n9"`

**Big O.** O(N). Calling `print` 10^5 times is several times slower than one write.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-15-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-15-7)

In [ ]:
# 15.7 Build output once
def squares_output(n):
    raise NotImplementedError

In [ ]:
def test_ex_15_7():
    assert squares_output(3) == '1\n4\n9'


test_ex_15_7()
print('15.7 passed')

### 15.8 · Iterative DFS with an explicit stack  `Core`

**Task.** Write `reachable(graph, start)` returning the set of nodes reachable from start using a list as a stack, so a path of 10^5 nodes does not hit the recursion limit.

**Example.** `reachable({1: [2], 2: [3], 3: [], 4: [1]}, 1)` -> `{1, 2, 3}`

**Big O.** O(V + E).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-15-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-15-8)

In [ ]:
# 15.8 Iterative DFS with an explicit stack
def reachable(graph, start):
    raise NotImplementedError

In [ ]:
def test_ex_15_8():
    assert reachable({1: [2], 2: [3], 3: [], 4: [1]}, 1) == {1, 2, 3}
    chain = {i: [i + 1] for i in range(200000)}
    assert len(reachable(chain, 0)) == 200001


test_ex_15_8()
print('15.8 passed')